# 🧬 SPS Self-Specialization — Dynamic Serialization + Capability Handler Demo

**Research claim:** the system starts with only `IntegerMultiplication [S0]`. When `FloatMultiplication` is missing, it dynamically serializes/generalizes the existing State 0 capability, creates `SerializeCapability [S0]`, reparents the existing integer capability without changing its ID, creates a transient `S0-C` copy, specializes the copy with Ollama/Qwen, verifies it, activates the result as `S1`, persists it, reloads it, and reuses it.

## 🔬 Complete research flow

```text
INITIAL
IntegerMultiplication [S0]
        │
        │ request multiply(2.5, 4.0)
        ▼
FloatMultiplication missing
        │
        ▼
SERIALIZE / GENERALIZE existing IntegerMultiplication [S0]
        │
        ▼
SerializeCapability [S0]  ← CREATED NOW
        │
        ├── IntegerMultiplication [S0]  ← SAME ID, reparented
        │
        ▼
REPLICATE → transient copy [S0-C]
        ↓
SPECIALIZE → Ollama + Qwen Coder
        ↓
GENERATED → FloatMultiplication
        ↓
VERIFY → syntax/policy + functional cases
        ↓
ACTIVATE → FloatMultiplication [S1]
        ↓
FINAL
              SerializeCapability [S0]
                        │
               ┌────────┴────────┐
               ▼                 ▼
 IntegerMultiplication    FloatMultiplication
        [S0]                    [S1]

PERSIST → RELOAD → REUSE without another AI call
```

**Important:** `SerializeCapability` does not exist before the float request, and creating it does not change the integer capability from S0 to S1.

## 1. Install and start Ollama first

**Run this cell first.** Ollama is started from `/content`, a stable directory that is not deleted when the repository is recloned.

In [ ]:
%cd /content
!apt-get update -qq
!apt-get install -y -qq zstd curl
!curl -fsSL https://ollama.com/install.sh | sh
!pkill -9 ollama || true
!pkill -9 llama-server || true
!nohup ollama serve >/tmp/ollama.log 2>&1 &
!sleep 5
!ollama --version
!curl -sf http://127.0.0.1:11434/api/tags || (cat /tmp/ollama.log; exit 1)
!ollama pull qwen2.5-coder:7b

## 2. Clone the latest `main` branch and install dependencies

In [ ]:
%cd /content
!rm -rf self-specialization
!git clone --branch main --single-branch -q https://github.com/muhammadnaumantahir/self-specialization.git
%cd /content/self-specialization
!pip -q install -r requirements.txt pytest
!echo 'Repository commit:'
!git rev-parse HEAD
!echo '\nAvailable Ollama models:'
!ollama list

## 3. Run deterministic tests before the real model

These tests validate dynamic serialization, reparenting, S0-C replication, specialization, verification, the final hierarchy, persistence and reload. They do not require Ollama.

In [ ]:
%cd /content/self-specialization
!PYTHONPATH=. pytest -q

## 4. Run the real SPS experiment

The demo creates a fresh registry. Its initial state contains only `IntegerMultiplication [S0]`.

In [ ]:
%cd /content/self-specialization
import os
os.environ['OLLAMA_MODEL'] = 'qwen2.5-coder:7b'
!PYTHONPATH=. python experiments/self_specialization_demo.py

## 5. What the demo must prove

### Initial state
```text
IntegerMultiplication [S0]
```
No `SerializeCapability`, `FloatMultiplication`, or permanent replication copy exists.

### Runtime evolution
1. Detect missing `[float, float] -> float`.
2. Select existing `IntegerMultiplication [S0]`.
3. Create `SerializeCapability [S0]` dynamically.
4. Reparent the same integer object/ID under the new S0 parent.
5. Replicate integer into transient `S0-C`.
6. Ask Qwen to specialize the copy into `FloatMultiplication`.
7. Verify the generated implementation.
8. Activate the verified result as `S1`.
9. Link float directly under SerializeCapability.

### Final hierarchy
```text
              SerializeCapability [S0]
                        │
               ┌────────┴────────┐
               ▼                 ▼
 IntegerMultiplication    FloatMultiplication
        [S0]                    [S1]
```

The original integer capability remains S0. The S0-C copy is transient and does not appear in the final hierarchy.

## 6. State model

| State | Meaning |
|---|---|
| `S0` | Original/general capability state |
| `S0-C` | Transient replicated copy used for specialization |
| `GENERATED` | Generated source before activation |
| `S1` | Verified and active specialized capability |
| `FAILED` | Generation or verification failed |

## 7. Supervisor checklist

- 🔵 Initial state: only `IntegerMultiplication [S0]`.
- 🟡 Float request: required typed capability is missing.
- 🧩 Serialization/generalization: `SerializeCapability [S0]` appears only now.
- 🔗 Reparent: original integer keeps its ID and remains S0.
- 🔁 Replication: transient `S0-C` copy is created.
- 🧠 Specialization: Ollama + `qwen2.5-coder:7b`.
- 🛡️ Verification gates activation.
- 🟢 Float becomes S1.
- 🌳 Final hierarchy has integer S0 and float S1 as siblings.
- 💾 Metadata/source are persisted.
- 🔄 Reload/reuse does not regenerate the float capability.

## 8. If Ollama fails

Run the diagnostic cell below.

In [ ]:
%cd /content
!pwd
!echo '\nOllama processes:'
!ps aux | grep -E 'ollama|llama-server' | grep -v grep || true
!echo '\nOllama API:'
!curl -s http://127.0.0.1:11434/api/tags || true
!echo '\nOllama log:'
!cat /tmp/ollama.log